# gpu-quantum-sim: Colab quickstart (free T4 GPU)

Runs the full project on a GPU: the test suite, the GHZ CPU-vs-GPU benchmark, the H₂ VQE, and the README refresh.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

This notebook contains no logic of its own. Every cell calls a script from the repository, so the results are identical to running `make all` on a GPU workstation.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Get the code

Set `REPO_URL` to your fork or clone of the repository.

In [ ]:
REPO_URL = "https://github.com/gracinting/gpu-quantum-sim.git"

import os
if not os.path.isdir("gpu-quantum-sim"):
    !git clone --depth 1 {REPO_URL} gpu-quantum-sim
%cd gpu-quantum-sim

## 3. Install the pinned dependencies

`cudaq` detects the runtime's CUDA version and installs the matching GPU wheel. NumPy, SciPy and Matplotlib are left at the versions Colab ships so the runtime does not need a restart; the quantum packages are pinned to the versions in `requirements.txt`.

In [ ]:
%pip install -q cudaq==0.16.0 pyscf==2.15.0 openfermion==1.8.1 openfermionpyscf==0.5 psutil pytest

In [ ]:
import cudaq
print(cudaq.__version__)
print("GPUs visible to CUDA-Q:", cudaq.num_available_gpus())
print("nvidia target available:", cudaq.has_target("nvidia"))

## 4. Run the tests

On a GPU runtime the GPU test runs instead of being skipped.

In [ ]:
!python -m pytest -q

## 5. GHZ benchmark: CPU vs. GPU

Sweeps n = 20…30. A 30-qubit single-precision statevector is 8 GiB, which fits in the T4's 16 GB; the CPU sweep stops on its own when a point is projected to exceed the time budget. Expect this cell to take several minutes, most of it on the CPU baseline.

In [ ]:
!python src/ghz_benchmark.py --max-qubits 30

## 6. H₂ VQE and bond-length scan

In [ ]:
!python src/h2_vqe.py

## 7. Refresh the README numbers from the new CSVs

In [ ]:
!python src/make_readme.py

## 8. Look at the results

In [ ]:
from IPython.display import Image, Markdown, display
import os

for name in ["ghz_benchmark.png", "ghz_speedup.png", "h2_pes.png", "h2_error.png", "vqe_convergence.png"]:
    path = os.path.join("results", name)
    if os.path.exists(path):
        display(Image(path, width=720))

display(Markdown(open("README.md").read().split("## Results")[0]))

In [ ]:
!cat results/ghz_benchmark.csv

## 9. Download the results

Unzip this over your local clone (it contains `results/` and the refreshed `README.md`), review the diff and commit.

In [ ]:
!zip -q -r gpu-quantum-sim-results.zip results README.md -x "results/quick/*"
from google.colab import files
files.download("gpu-quantum-sim-results.zip")